In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\Sajib
[nltk_data]     Saha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Sajib
[nltk_data]     Saha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Sajib
[nltk_data]     Saha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Sajib
[nltk_data]     Saha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv("Twitter_Data.csv")

In [3]:
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [6]:
df.isna().sum()

clean_text    4
category      7
dtype: int64

In [7]:
df.shape

(162980, 2)

In [8]:
df= df.dropna()

In [9]:
import re

# stopwords
stop_w = set(stopwords.words('english'))

# lemmatizer
lemmatizer = WordNetLemmatizer()

def text_preprocessing(phrase): # class e convert kore niben

    # lowercase
    phrase = phrase.lower()

    # contractions
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can't", "can not", phrase)

    # general contractions
    phrase = re.sub(r"n't", " not", phrase)
    phrase = re.sub(r"'re", " are", phrase)
    phrase = re.sub(r"'s", " is", phrase)
    phrase = re.sub(r"'d", " would", phrase)
    phrase = re.sub(r"'ll", " will", phrase)
    phrase = re.sub(r"'ve", " have", phrase)
    phrase = re.sub(r"'m", " am", phrase)

    # custom replacement
    phrase = re.sub(r"luv", "love", phrase)

    # remove emoji, punctuation, digits
    phrase = re.sub(r"[^a-zA-Z\s]", " ", phrase)

    # remove extra spaces
    phrase = re.sub(r"\s+", " ", phrase).strip()

    # tokenize
    words = phrase.split()

    # stopword removal + lemmatization
    words = [
        lemmatizer.lemmatize(word.lower())
        for word in words
        if word not in stop_w
    ]

    return " ".join(words)

# apply
df["text"] = df["clean_text"].apply(text_preprocessing)

In [10]:
df.head()

,clean_text,category,text
0,when modi promised “minimum government maximum...,-1.0,modi promised minimum government maximum gover...
1,talk all the nonsense and continue all the dra...,0.0,talk nonsense continue drama vote modi
2,what did just say vote for modi welcome bjp t...,1.0,say vote modi welcome bjp told rahul main camp...
3,asking his supporters prefix chowkidar their n...,1.0,asking supporter prefix chowkidar name modi gr...
4,answer who among these the most powerful world...,1.0,answer among powerful world leader today trump...


In [25]:
x = df[["text"]]
y = df[["category"]]

In [26]:
split_range= .80
len_x = len(x)
split_num = int(split_range*len_x)
train_text = x[:split_num]
train_label = y[:split_num]

test_text = x[split_num:]
test_label= y[split_num:]

In [27]:
test_text

,text
130376,pradhan mantri mudra yojana crore people got m...
130377,modi everywhere like
130378,modi gatbandhan cbi medium calling name genuin...
130379,attacking congress modi said grand old party s...
130380,ill chowkidaar power modi realized chowkidaar ...
...,...
162975,crore paid neerav modi recovered congress lead...
162976,dear rss terrorist payal gawar modi killing pl...
162977,cover interaction forum left
162978,big project came india modi dream project happ...


In [28]:
test_label

,category
130376,-1.0
130377,0.0
130378,1.0
130379,1.0
130380,-1.0
...,...
162975,-1.0
162976,-1.0
162977,0.0
162978,0.0


In [29]:
from gensim.models import Word2Vec, KeyedVectors

embedding_size= 256
model = Word2Vec(train_label, vector_size= embedding_size, window=7, min_count=1, workers=4)

In [30]:
import numpy as np

def getVectors(dataset):
    vectors = []

    for dataItem in dataset:
        singleDataItemEmbedding = np.zeros(embedding_size)
        wordCount = 0

        for word in dataItem:
            if word in model.wv.key_to_index:
                singleDataItemEmbedding += model.wv[word]
                wordCount += 1

        if wordCount > 0:
            singleDataItemEmbedding /= wordCount

        vectors.append(singleDataItemEmbedding)

    return np.array(vectors)


trainReviewVectors = getVectors(train_text)
testReviewVectors = getVectors(test_text)

In [31]:
trainReviewVectors

array([[-3.10108097e-03,  2.08082547e-03, -1.70587096e-04,
         2.12822901e-03, -4.95505519e-04,  3.77608618e-04,
         1.51215432e-03, -9.58851073e-04, -1.15663745e-04,
         1.42380176e-03,  1.51865355e-03, -6.02365471e-05,
         1.98952435e-03, -2.23457441e-03,  5.41443005e-05,
         8.82982276e-05,  2.07017936e-03,  1.48348200e-03,
         9.59355850e-04,  3.00509591e-03, -3.33225681e-03,
         1.54754799e-03, -2.38263762e-03, -8.86252150e-04,
        -2.27194466e-03, -1.20249375e-03, -2.78298091e-03,
        -2.19306163e-03, -2.14473112e-03, -7.68431152e-05,
        -9.02705515e-04,  1.59608666e-03, -3.18770794e-03,
        -1.60680929e-03,  3.38010692e-03,  2.30715067e-03,
         8.42523761e-04,  6.58323988e-04,  4.38579048e-04,
        -1.27920151e-03,  3.35097158e-04,  1.28140983e-03,
         8.77847585e-04,  7.03205355e-04,  2.52190791e-03,
         1.80258524e-03, -2.11289572e-03, -6.01213425e-04,
        -1.17079820e-04,  2.89075930e-03,  2.14305598e-0